# 01. 데이터 전처리

국제 원유/석유제품 가격, 환율, 국내 평균 소매가, 브랜드별 가격, 유류세, 정유사 주간 공급가격을 하나의 일별 분석용 데이터로 통합합니다.

이 노트북은 Google Colab에서 단독 실행되도록 구성되어 있습니다. 첫 설정 셀의 경로만 본인 Google Drive 구조에 맞게 수정한 뒤 위에서 아래로 실행하면 됩니다.


In [ ]:
# ============================================================
# 01 공통 경로 설정
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"
DATA_COLLECTION_PATH = os.path.join(ROOT_PATH, "data-analysis", "00_data_collection", "outputs") + "/"
PROCESSED_PATH = os.path.join(ROOT_PATH, "preprocessed_data") + "/"

print(f"ROOT_PATH           = {ROOT_PATH}")
print(f"DATA_COLLECTION_PATH= {DATA_COLLECTION_PATH}")
print(f"PROCESSED_PATH      = {PROCESSED_PATH}")

# 필요한 패키지 확인/설치
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "xlrd": "xlrd",
    "openpyxl": "openpyxl",
    "lxml": "lxml",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print(f"[설치] 필요한 패키지 설치: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("[확인] 필요한 패키지가 이미 설치되어 있습니다.")

In [ ]:
# =========================
# 1) 입력 파일 경로 설정
# =========================

import os
import re
from pathlib import Path
from typing import Iterable, List, Optional
import calendar
import numpy as np
import pandas as pd


def latest_under(base, pattern):
    base = Path(base)
    if not base.exists():
        return None
    files = [p for p in base.glob(pattern) if p.exists() and p.is_file()]
    if not files:
        return None
    return sorted(files, key=lambda p: (p.stat().st_mtime, str(p)), reverse=True)[0]


def resolve_collected_input(dataset, pattern):
    dataset_dir = Path(DATA_COLLECTION_PATH) / dataset
    collected = latest_under(dataset_dir, pattern)
    if collected is not None:
        print(f"[INPUT] {dataset}: {collected}")
        return str(collected)

    return str(dataset_dir / pattern)


CRUDE_PATH = resolve_collected_input("crude", "crude_*.csv")
RETAIL_AVG_PATH = resolve_collected_input("retail_avg", "retail_avg_*.csv")
BRAND_GASOLINE_PATH = resolve_collected_input("brand_price", "brand_gasoline_*.csv")
BRAND_DIESEL_PATH = resolve_collected_input("brand_price", "brand_diesel_*.csv")
FX_PATH = resolve_collected_input("fx_usdkrw", "fx_usdkrw_*.csv")
INTL_PRODUCTS_PATH = resolve_collected_input("intl_products", "intl_products_*.csv")
DIESEL_TAX_TREND_PATH = resolve_collected_input("fuel_tax_trend", "diesel_tax_trend_*.xls")
GASOLINE_TAX_TREND_PATH = resolve_collected_input("fuel_tax_trend", "gasoline_tax_trend_*.xls")
REFINERY_WEEKLY_SUPPLY_PRICES_BY_PRODUCT_PATH = resolve_collected_input("refinery_weekly_supply", "refinery_weekly_supply_prices_by_product_*.csv")
INTL_DIESEL001_PATH = resolve_collected_input("intl_products", "intl_product_diesel(0.001)_*.csv")

BARREL_TO_LITER = 158.987294928

In [ ]:
# =========================
# 2) 유틸 함수
# =========================
def ensure_dir(path: str) -> None:
    """
    입력: path: str
    출력: None
    설명: 지정한 경로의 디렉터리가 없으면 상위 폴더까지 포함해 생성
    """

    Path(path).mkdir(parents=True, exist_ok=True)


def parse_kor_date(x: str):
    """
    입력: x: str
    출력: pd.Timestamp
    설명: 한국어 날짜 문자열 또는 YYYY/MM/DD 형식 문자열을 Timestamp로 변환
    """

    s = str(x).replace("\xa0", "").replace(" ", "").strip()

    m = re.match(r"(\d{2,4})년(\d{1,2})월(\d{1,2})일", s)
    if m:
        y = int(m.group(1))
        if y < 100:
            y += 2000 if y < 50 else 1900
        return pd.Timestamp(y, int(m.group(2)), int(m.group(3)))

    m = re.match(r"(\d{4})/(\d{1,2})/(\d{1,2})", s)
    if m:
        return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

    raise ValueError(f"날짜 파싱 실패: {x}")


def coerce_numeric(
    df: pd.DataFrame,
    exclude: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    """
    입력: df: pd.DataFrame, exclude: Optional[Iterable[str]]=None
    출력: pd.DataFrame
    설명: 제외 컬럼을 빼고 나머지 컬럼을 숫자형으로 변환
    """

    exclude = set(exclude or [])
    out = df.copy()
    for c in out.columns:
        if c in exclude:
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


def save_csv(df: pd.DataFrame, path: str) -> None:
    """
    입력: df: pd.DataFrame, path: str
    출력: None
    설명: 데이터프레임을 utf-8-sig 인코딩 CSV로 저장하고 로그를 출력
    """

    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[로그] CSV 저장 완료: {path}")


def read_csv_robust(
    path: str,
    encodings: Optional[List[str]] = None,
    **kwargs,
) -> pd.DataFrame:

    """
    입력: path: str, encodings: Optional[List[str]]=None, **kwargs
    출력: pd.DataFrame
    설명: 여러 인코딩을 순차적으로 시도하여 CSV를 안정적으로 읽음
    """
    encodings = encodings or ["utf-8-sig", "cp949", "euc-kr", "utf-8"]
    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except Exception as e:
            last_error = e

    raise last_error

def read_excel_robust(
    path: str,
    sheet_name=0,
    header=0,
    engine: Optional[str] = None,
    **kwargs,
) -> pd.DataFrame:
    """
    입력: path: str, sheet_name=0, header=0, engine: Optional[str]=None, **kwargs
    출력: pd.DataFrame
    설명: 여러 엔진과 HTML fallback을 순차적으로 시도하여 Excel 파일을 안정적으로 읽어 반환
    """

    if not os.path.exists(path):
        raise FileNotFoundError(f"엑셀 파일이 없습니다: {path}")

    ext = os.path.splitext(path)[1].lower()

    if engine is not None:
        return pd.read_excel(
            path,
            sheet_name=sheet_name,
            header=header,
            engine=engine,
            **kwargs,
        )

    if ext == ".xls":
        engines = ["xlrd", "openpyxl"]
    elif ext in [".xlsx", ".xlsm", ".xltx", ".xltm"]:
        engines = ["openpyxl", "xlrd"]
    else:
        engines = ["openpyxl", "xlrd"]

    last_error = None

    for eng in engines:
        try:
            return pd.read_excel(
                path,
                sheet_name=sheet_name,
                header=header,
                engine=eng,
                **kwargs,
            )
        except Exception as e:
            last_error = e

    try:
        tables = pd.read_html(path, header=header)
        if len(tables) == 0:
            raise ValueError("HTML 테이블이 없습니다.")
        return tables[sheet_name if isinstance(sheet_name, int) else 0]
    except Exception as e:
        raise RuntimeError(
            f"Excel 읽기 실패: {path}\n"
            f"시도한 엔진: {engines} + html fallback\n"
            f"마지막 오류(read_excel): {last_error}\n"
            f"마지막 오류(read_html): {e}"
        )

In [ ]:
# =========================
# 3) 원천데이터 로딩 함수
# =========================
def read_crude(path: str) -> pd.DataFrame:
    """
    입력: path: str
    출력: pd.DataFrame
    설명: 원유 가격 CSV를 읽어 날짜를 변환하고 주요 유종 컬럼명을 표준화한 뒤 정렬하여 반환
    """

    df = read_csv_robust(path)
    df["date"] = df["기간"].apply(parse_kor_date)
    df = df.drop(columns=["기간"]).rename(
        columns={
            "Dubai": "두바이",
            "Brent": "브렌트",
            "WTI": "WTI",
        }
    )
    return coerce_numeric(df, exclude=["date"]).sort_values("date")


def read_retail_avg(path: str) -> pd.DataFrame:
    """
    입력: path: str
    출력: pd.DataFrame
    설명: 국내 평균 소매가 CSV를 읽어 날짜를 변환하고 휘발유·경유 평균 컬럼명을 정리하여 반환
    """

    df = read_csv_robust(path)
    df["date"] = df["구분"].apply(parse_kor_date)
    df = df.drop(columns=["구분"]).rename(
        columns={
            "보통휘발유": "보통휘발유_평균",
            "자동차용경유": "자동차용경유_평균",
        }
    )
    return coerce_numeric(df, exclude=["date"]).sort_values("date")


def read_brand(path: str, fuel_prefix: str) -> pd.DataFrame:
    """
    입력: path: str, fuel_prefix: str
    출력: pd.DataFrame
    설명: 브랜드별 가격 CSV를 읽어 날짜를 변환하고 각 브랜드 컬럼 앞에 연료명 접두사를 붙여 반환
    """

    df = read_csv_robust(path)
    df["date"] = df["구분"].apply(parse_kor_date)
    df = df.drop(columns=["구분"]).rename(columns={"(알뜰-자영)": "알뜰-자영"})

    rename_map = {}
    for c in df.columns:
        if c == "date":
            continue
        rename_map[c] = f"{fuel_prefix}_{c}"

    df = df.rename(columns=rename_map)
    return coerce_numeric(df, exclude=["date"]).sort_values("date")


def read_fx(path: str) -> pd.DataFrame:
    """
    입력: path: str
    출력: pd.DataFrame
    설명: 환율 CSV를 읽어 날짜와 원/달러 환율 컬럼만 정리하여 반환
    """

    df = read_csv_robust(path)
    df["date"] = df["변환"].apply(parse_kor_date)
    df["usdkrw"] = (
        df["원자료"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace('"', "", regex=False)
        .str.strip()
    )
    df["usdkrw"] = pd.to_numeric(df["usdkrw"], errors="coerce")
    return df[["date", "usdkrw"]].sort_values("date")




def read_intl_products(path: str) -> pd.DataFrame:
    """
    입력: path: str
    출력: pd.DataFrame
    설명: 국제제품가 CSV를 읽어 날짜를 변환하고 제품 컬럼명을 분석용 표준명으로 정리해 반환
    """

    df = read_csv_robust(path)
    df["date"] = df["기간"].apply(parse_kor_date)
    out = df.drop(columns=["기간"]).rename(columns={
        "휘발유[95RON]": "휘발유95RON",
        "휘발유(95RON)": "휘발유95RON",
        "휘발유[92RON]": "휘발유92RON",
        "휘발유(92RON)": "휘발유92RON",
        "경유[0.05%]": "경유0.05",
        "경유(0.05%)": "경유0.05",
        "경유[0.001%]": "경유0.001",
        "경유(0.001%)": "경유0.001",
    })
    out["date"] = df["date"]
    out = coerce_numeric(out, exclude=["date"]).sort_values("date")

    out["non_nulls"] = out.drop(columns=["date"]).notna().sum(axis=1)
    out = (
        out.sort_values(["date", "non_nulls"])
        .drop_duplicates("date", keep="last")
        .drop(columns=["non_nulls"])
    )

    return out


def read_tax(path: str) -> pd.DataFrame:
    """
    입력: path: str
    출력: pd.DataFrame
    설명: 유류세 추이 Excel/HTML 파일을 읽어 0번 행을 열 이름으로 사용하고,
         합계 컬럼을 제거한 뒤 날짜를 일 단위로 확장하여 변동 전 값을 forward fill하여 반환
    """

    df = read_excel_robust(path)

    df.columns = df.iloc[0]
    df = df.iloc[1:].copy()

    df = df.drop(columns=["합계"], errors="ignore")

    df["date"] = df["변동일자"].apply(parse_kor_date)

    df = df.drop(columns=["변동일자"])
    df = coerce_numeric(df, exclude=["date"]).sort_values("date").reset_index(drop=True)

    calendar = pd.DataFrame({
        "date": pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    })

    out = calendar.merge(df, on="date", how="left").sort_values("date")

    value_cols = [c for c in out.columns if c != "date"]
    out[value_cols] = out[value_cols].ffill()

    return out.reset_index(drop=True)

def read_refinery(path: str, fuel_col: str) -> pd.DataFrame:
    """
    입력: path: str, fuel_col: str
    출력: pd.DataFrame
    설명: 정유사 주간 공급가격 CSV를 읽어 지정한 연료 컬럼만 선택하고,
         주차별 마지막 날짜를 date로 변환하여 반환
    """

    df = read_csv_robust(path)

    if fuel_col not in df.columns:
        raise ValueError(f"지정한 연료 컬럼이 없습니다: {fuel_col}")

    def parse_week_end(x: str) -> pd.Timestamp:
        """
        입력: x: str
        출력: pd.Timestamp
        설명: '08년05월1주' 형태 문자열을 해당 월-주차의 마지막 날짜로 변환
        """

        s = str(x).replace("\xa0", "").replace(" ", "").strip()
        m = re.match(r"(\d{2})년(\d{2})월(\d)주", s)
        if not m:
            raise ValueError(f"주차 파싱 실패: {x}")

        yy = int(m.group(1))
        mm = int(m.group(2))
        ww = int(m.group(3))

        year = 2000 + yy if yy < 50 else 1900 + yy
        last_day = calendar.monthrange(year, mm)[1]

        if ww == 1:
            end_day = min(7, last_day)
        elif ww == 2:
            end_day = min(14, last_day)
        elif ww == 3:
            end_day = min(21, last_day)
        elif ww == 4:
            end_day = min(28, last_day)
        elif ww == 5:
            end_day = last_day
        else:
            raise ValueError(f"주차 값이 올바르지 않습니다: {x}")

        return pd.Timestamp(year, mm, end_day)

    out_col_name = f"정유소_세전_{fuel_col}"

    df["date"] = df["구분"].apply(parse_week_end)
    df = df.rename(columns={fuel_col: out_col_name})

    return (
        coerce_numeric(df[["date", out_col_name]], exclude=["date"])
        .sort_values("date")
        .drop_duplicates("date", keep="last")
        .reset_index(drop=True)
    )

def read_diesel001(path: str) -> pd.DataFrame:
    """
    입력: path: str
    출력: pd.DataFrame
    설명:
    """

    df = read_csv_robust(path)

    df["date"] = df["기간"].apply(parse_kor_date)

    df = df.drop(columns=["기간"])
    df = coerce_numeric(df, exclude=["date"]).sort_values("date").reset_index(drop=True)
    calendar = pd.DataFrame({
        "date": pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    })

    out = calendar.merge(df, on="date", how="left").sort_values("date")

    out = out.rename(columns={"경유(0.001%)": "경유0.001",})

    value_cols = [c for c in out.columns if c != "date"]
    out[value_cols] = out[value_cols].ffill()

    return out.reset_index(drop=True)


In [ ]:
# =========================
# 4) 전처리 + 통합 파일 생성
# =========================
def main() -> pd.DataFrame:
    """
    입력: 없음
    출력: pd.DataFrame
    설명: 원천데이터를 로딩·전처리하여 분석용 일별 통합데이터 CSV를 생성
    """

    result_prep = PROCESSED_PATH
    ensure_dir(result_prep)

    # 입력 파일 확인
    input_paths = {
        "crude.csv": CRUDE_PATH,
        "retail_avg.csv": RETAIL_AVG_PATH,
        "brand_gasoline.csv": BRAND_GASOLINE_PATH,
        "brand_diesel.csv": BRAND_DIESEL_PATH,
        "fx_usdkrw.csv": FX_PATH,
        "intl_products.csv": INTL_PRODUCTS_PATH,
        "intl_product_diesel(0.001).csv": INTL_DIESEL001_PATH,
        "diesel_tax_trend.xls": DIESEL_TAX_TREND_PATH,
        "gasoline_tax_trend.xls": GASOLINE_TAX_TREND_PATH,
        "refinery_weekly_supply_prices_by_product.csv": REFINERY_WEEKLY_SUPPLY_PRICES_BY_PRODUCT_PATH,
    }
    missing = {name: p for name, p in input_paths.items() if not os.path.exists(p)}
    if missing:
        print("[오류] data-analysis/00_data_collection/outputs에서 찾지 못한 필수 입력 파일:")
        for name, p in missing.items():
            print(f"  - {name}: {p}")
        raise FileNotFoundError("필수 입력 파일이 없습니다. 위 목록을 확인하세요.")

    # 원천데이터 로딩
    crude = read_crude(CRUDE_PATH)
    retail_avg = read_retail_avg(RETAIL_AVG_PATH)
    brand_g = read_brand(BRAND_GASOLINE_PATH, "보통휘발유")
    brand_d = read_brand(BRAND_DIESEL_PATH, "자동차용경유")
    fx = read_fx(FX_PATH)
    intl_prod = read_intl_products(INTL_PRODUCTS_PATH)
    diesel001 = read_diesel001(INTL_DIESEL001_PATH)

    tax_g = read_tax(GASOLINE_TAX_TREND_PATH)
    tax_d = read_tax(DIESEL_TAX_TREND_PATH)

    refinery_g = read_refinery(
        REFINERY_WEEKLY_SUPPLY_PRICES_BY_PRODUCT_PATH,
        "보통휘발유",
    )
    refinery_d = read_refinery(
        REFINERY_WEEKLY_SUPPLY_PRICES_BY_PRODUCT_PATH,
        "자동차용경유",
    )

    # 세금 컬럼명 충돌 방지용 접두어 부여
    tax_g = tax_g.rename(
        columns={c: f"보통휘발유_{c}" for c in tax_g.columns if c != "date"}
    )
    tax_d = tax_d.rename(
        columns={c: f"자동차용경유_{c}" for c in tax_d.columns if c != "date"}
    )

    # 국내 데이터 통합
    domestic = (
        retail_avg
        .merge(brand_g, on="date", how="left")
        .merge(brand_d, on="date", how="left")
        .merge(tax_g, on="date", how="left")
        .merge(tax_d, on="date", how="left")
        .merge(refinery_g, on="date", how="left")
        .merge(refinery_d, on="date", how="left")
        .sort_values("date")
    )

    start_date = domestic["date"].min()
    end_date_full = domestic["date"].max()
    end_date_fx = fx["date"].max()

    # 날짜 캘린더 생성 후 국제지표 결합
    calendar = pd.DataFrame({
        "date": pd.date_range(start_date, end_date_full, freq="D")
    })

    product_keep = [
        "date",
        "휘발유92RON",
        "경유0.05",
    ]
    product_keep = [c for c in product_keep if c in intl_prod.columns]

    bench = (
        calendar
        .merge(crude, on="date", how="left")
        .merge(intl_prod[product_keep], on="date", how="left")
        .merge(diesel001, on="date", how="left")
        .merge(fx, on="date", how="left")
        .sort_values("date")
    )

    # 비영업일 값 보간(forward fill)
    ffill_cols = [
        "두바이", "브렌트", "WTI", "휘발유92RON", "경유0.05", "경유0.001",
    ]
    ffill_cols = [c for c in ffill_cols if c in bench.columns]

    for c in ffill_cols:
        bench[c] = bench[c].ffill()

    bench["usdkrw"] = bench["usdkrw"].ffill()

    # USD/bbl -> KRW/L 환산
    krw_cols = [
        "두바이", "브렌트", "WTI",
        "휘발유95RON", "휘발유92RON",
        "경유0.05", "경유0.001", "나프타",
    ]
    krw_cols = [c for c in krw_cols if c in bench.columns]

    for c in krw_cols:
        bench[f"{c}_원리터"] = bench[c] * bench["usdkrw"] / BARREL_TO_LITER

    # 환율이 존재하는 마지막 날짜까지만 사용
    bench_main = bench[bench["date"] <= end_date_fx].copy()
    dom_main = domestic[domestic["date"] <= end_date_fx].copy()

    # 최종 통합 파일
    processed_daily = (
        bench_main
        .merge(dom_main, on="date", how="inner")
        .sort_values("date")
        .reset_index(drop=True)
    )

    output_file = os.path.join(result_prep, "분석용_일별_통합데이터.csv")
    save_csv(processed_daily, output_file)

    print("[로그] 전처리 완료")
    print(f"[로그] 출력 파일: {output_file}")
    print(f"[로그] 행/열 수: {processed_daily.shape[0]:,} rows x {processed_daily.shape[1]:,} columns")
    print(f"[로그] 날짜 범위: {processed_daily["date"].min().date()} ~ {processed_daily["date"].max().date()}")

    return processed_daily

In [ ]:
# =========================
# 5) 실행
# =========================
processed_daily = main()
processed_daily.head()
